In [1]:
pip install rapidfuzz

  Using cached rapidfuzz-3.12.2-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (12 kB)
Using cached rapidfuzz-3.12.2-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (3.1 MB)
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import geopandas as gpd
import folium
from geopy.distance import geodesic
from rapidfuzz import process, fuzz
import matplotlib.pyplot as plt
import numpy as np

In [3]:
df = pd.read_csv('eladownload2024.csv')

/tmp/ipykernel_172/1165263448.py:1: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('eladownload2024.csv')


In [4]:

df_clean = df.dropna(subset=['countyname'])
schools = df_clean[df_clean['countyname'].str.contains('San Francisco')]

In [5]:
schools

,cds,rtype,schoolname,districtname,countyname,charter_flag,coe_flag,dass_flag,studentgroup,currdenom,...,currdenom_withoutPRLOSS,currstatus_withoutPRLOSS,priorprate_enrolled,priorprate_tested,priorprate,priornumPRLOSS,priordenom_withoutPRLOSS,priorstatus_withoutPRLOSS,indicator,reportingyear
132167,38103890000000,D,NaN,San Francisco County Office of Education,San Francisco,NaN,Y,NaN,AA,22,...,2,NaN,15.0,3.0,20.0,12.0,1,NaN,ELA,2024
132168,38103890000000,D,NaN,San Francisco County Office of Education,San Francisco,NaN,Y,NaN,ALL,163,...,17,-142.8,71.0,19.0,27.0,49.0,12,-149.6,ELA,2024
132169,38103890000000,D,NaN,San Francisco County Office of Education,San Francisco,NaN,Y,NaN,AS,16,...,2,NaN,5.0,2.0,40.0,3.0,1,NaN,ELA,2024
132170,38103890000000,D,NaN,San Francisco County Office of Education,San Francisco,NaN,Y,NaN,EL,38,...,4,NaN,19.0,8.0,42.0,10.0,5,NaN,ELA,2024
132171,38103890000000,D,NaN,San Francisco County Office of Education,San Francisco,NaN,Y,NaN,ELO,33,...,3,NaN,18.0,8.0,44.0,10.0,5,NaN,ELA,2024
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
134197,38771310137307,S,KIPP Bayview Elementary,SBE - KIPP Bayview Elementary,San Francisco,Y,NaN,NaN,RFP,2,...,2,NaN,2.0,2.0,100.0,NaN,2,NaN,ELA,2024
134198,38771310137307,S,KIPP Bayview Elementary,SBE - KIPP Bayview Elementary,San Francisco,Y,NaN,NaN,SBA,45,...,45,-67.5,NaN,NaN,NaN,NaN,56,-110.6,ELA,2024
134199,38771310137307,S,KIPP Bayview Elementary,SBE - KIPP Bayview Elementary,San Francisco,Y,NaN,NaN,SED,43,...,43,-68.3,51.0,51.0,100.0,NaN,50,-109.1,ELA,2024
134200,38771310137307,S,KIPP Bayview Elementary,SBE - KIPP Bayview Elementary,San Francisco,Y,NaN,NaN,SWD,11,...,11,-117.4,11.0,11.0,100.0,NaN,11,-162.1,ELA,2024


In [6]:
highschools = schools[~schools['schoolname'].str.contains('Elementary|Middle', case=False, na=False)]
highschools

,cds,rtype,schoolname,districtname,countyname,charter_flag,coe_flag,dass_flag,studentgroup,currdenom,...,currdenom_withoutPRLOSS,currstatus_withoutPRLOSS,priorprate_enrolled,priorprate_tested,priorprate,priornumPRLOSS,priordenom_withoutPRLOSS,priorstatus_withoutPRLOSS,indicator,reportingyear
132167,38103890000000,D,NaN,San Francisco County Office of Education,San Francisco,NaN,Y,NaN,AA,22,...,2,NaN,15.0,3.0,20.0,12.0,1,NaN,ELA,2024
132168,38103890000000,D,NaN,San Francisco County Office of Education,San Francisco,NaN,Y,NaN,ALL,163,...,17,-142.8,71.0,19.0,27.0,49.0,12,-149.6,ELA,2024
132169,38103890000000,D,NaN,San Francisco County Office of Education,San Francisco,NaN,Y,NaN,AS,16,...,2,NaN,5.0,2.0,40.0,3.0,1,NaN,ELA,2024
132170,38103890000000,D,NaN,San Francisco County Office of Education,San Francisco,NaN,Y,NaN,EL,38,...,4,NaN,19.0,8.0,42.0,10.0,5,NaN,ELA,2024
132171,38103890000000,D,NaN,San Francisco County Office of Education,San Francisco,NaN,Y,NaN,ELO,33,...,3,NaN,18.0,8.0,44.0,10.0,5,NaN,ELA,2024
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
134183,38769270132183,S,The New School of San Francisco,SBE - The New School of San Francisco,San Francisco,Y,NaN,NaN,RFP,25,...,25,39.6,28.0,28.0,100.0,NaN,27,7.8,ELA,2024
134184,38769270132183,S,The New School of San Francisco,SBE - The New School of San Francisco,San Francisco,Y,NaN,NaN,SBA,312,...,312,47.6,NaN,NaN,NaN,NaN,293,35.9,ELA,2024
134185,38769270132183,S,The New School of San Francisco,SBE - The New School of San Francisco,San Francisco,Y,NaN,NaN,SED,107,...,107,-10.6,100.0,99.0,99.0,NaN,98,-12.7,ELA,2024
134186,38769270132183,S,The New School of San Francisco,SBE - The New School of San Francisco,San Francisco,Y,NaN,NaN,SWD,44,...,44,-35.8,51.0,50.0,98.0,NaN,50,-31.3,ELA,2024


In [7]:
onlyhighschools = pd.read_csv('public_high_schools.csv')

In [8]:
onlyhighschools


,Campus Name,CCSF Entity,Lower Grade,Upper Grade,Grade Range,Category,Map Label,Lower Age,Upper Age,General Type,...,Campus Address,Supervisor District,County FIPS,County Name,Location 1,Neighborhoods (old),Zip Codes,Fire Prevention Districts,Police Districts,Supervisor Districts
0,"Marshall, Thurgood Marshall High School",SFUSD,9,12,9-12,USD Grades 9-12,PS073,14,17,PS,...,"45 CONKLING ST, San Francisco, CA 94124",10,6075,SAN FRANCISCO,"CA\n(37.736309, -122.401649)",1,58,10.0,3.0,8
1,"Hearst, Phoebe Apperson Hearst Home",SFUSD,9,12,9-12,USD Grades 9-12,PS045,14,17,PS,...,"3045 SANTIAGO ST, SAN FRANCISCO 94116",4,6075,SAN FRANCISCO,"CA\n(37.74363, -122.500053)",35,29491,1.0,8.0,3
2,"Burton, Phillip And Sala Burton High School",SFUSD,9,12,9-12,USD Grades 9-12,PS011,14,17,PS,...,"400 MANSELL ST, San Francisco, CA 94134",9,6075,SAN FRANCISCO,"CA\n(37.721546, -122.406555)",28,309,10.0,3.0,7
3,"Washington, George Washington High School",SFUSD,9,12,9-12,USD Grades 9-12,PS121,14,17,PS,...,"600 32ND AVE, San Francisco, CA 94121",1,6075,SAN FRANCISCO,"CA\n(37.777905, -122.491013)",26,55,11.0,6.0,2
4,"Lincoln, Abraham Lincoln High School",SFUSD,9,12,9-12,USD Grades 9-12,PS067,14,17,PS,...,"2162 24TH AVE, San Francisco, CA 94116",4,6075,SAN FRANCISCO,"CA\n(37.746594, -122.48024)",35,29491,1.0,8.0,3
5,Life Learning Academy Charter School,SFUSD,9,12,9-12,USD Charter School,PS064,14,17,PS,...,"651 8TH TI ST, SAN FRANCISCO, CA 94130",6,6075,SAN FRANCISCO,"CA\n(37.825512, -122.367996)",37,62,NaN,2.0,9
6,Gateway High School / Kipp Sf Bay Academy,SFUSD,9,12,9-12,USD Charter School,PS037,14,17,PS,...,"1430 SCOTT ST, San Francisco, CA 94115",5,6075,SAN FRANCISCO,"CA\n(37.783264, -122.436691)",41,29490,13.0,5.0,11
7,Galileo High School,SFUSD,9,12,9-12,USD Grades 9-12,PS035,14,17,PS,...,"1150 FRANCISCO ST, San Francisco, CA 94109",2,6075,SAN FRANCISCO,"CA\n(37.803791, -122.424149)",32,28858,5.0,9.0,1
8,Balboa High School,SFUSD,9,12,9-12,USD Grades 9-12,PS007,14,17,PS,...,"1000 CAYUGA AVE, San Francisco, CA 94112",11,6075,SAN FRANCISCO,"CA\n(37.721142, -122.441399)",25,28861,9.0,7.0,6
9,City Arts And Tech High School,SFUSD,9,12,9-12,USD Grades 9-12,PS019,14,17,PS,...,"325 LA GRANDE AVE, San Francisco, CA 94112",11,6075,SAN FRANCISCO,"CA\n(37.718784, -122.424667)",18,309,9.0,7.0,6


In [9]:
# Function to find similar names with debug
def find_similar_names(name, name_list, threshold=51):
    matches = process.extract(name, name_list, scorer=fuzz.ratio, limit=1)
    if matches:
        print(f"Checking '{name}' against '{matches[0][0]}' with score {matches[0][1]}")
        if matches[0][1] >= threshold:
            return True
    return False

In [10]:
names = onlyhighschools['Campus Name'].tolist()

In [11]:
similar_names = highschools['schoolname'].apply(lambda x: find_similar_names(x, names))

Checking 'S.F. County Court Woodside Learning Ctr' against 'Life Learning Academy Charter School' with score 40.0
Checking 'S.F. County Court Woodside Learning Ctr' against 'Life Learning Academy Charter School' with score 40.0
Checking 'S.F. County Court Woodside Learning Ctr' against 'Life Learning Academy Charter School' with score 40.0
Checking 'S.F. County Court Woodside Learning Ctr' against 'Life Learning Academy Charter School' with score 40.0
Checking 'S.F. County Court Woodside Learning Ctr' against 'Life Learning Academy Charter School' with score 40.0
Checking 'S.F. County Court Woodside Learning Ctr' against 'Life Learning Academy Charter School' with score 40.0
Checking 'S.F. County Court Woodside Learning Ctr' against 'Life Learning Academy Charter School' with score 40.0
Checking 'S.F. County Opportunity (Hilltop)' against 'City Arts And Tech High School' with score 34.92063492063492
Checking 'S.F. County Opportunity (Hilltop)' against 'City Arts And Tech High School' w

In [12]:
filtered_df_based_on_similar_names =highschools[similar_names]

In [13]:
filtered_df_based_on_similar_names['schoolname'].unique()

array(['Jordan (June) School for Equity',
       'City Arts & Leadership Academy',
       "Five Keys Independence HS (SF Sheriff's)",
       'S.F. International High', 'San Francisco Public Montessori',
       'Wells (Ida B.) High', 'Downtown High', 'Independence High',
       'Wallenberg (Raoul) Traditional High',
       'Burton (Phillip and Sala) Academic High', 'Balboa High',
       'Marshall (Thurgood) High', 'Life Learning Academy Charter',
       'Gateway High', 'Galileo High', 'Lincoln (Abraham) High',
       'Mission High', "O'Connell (John) High",
       'Washington (George) High', 'San Francisco Community Alternative'],
      dtype=object)

In [14]:
schools_to_remove = [
    'San Francisco Public Montessori',
    'San Francisco Community Alternative'
]

In [15]:
filtered_df_based_on_similar_names = filtered_df_based_on_similar_names[
    ~filtered_df_based_on_similar_names['schoolname'].isin(schools_to_remove)
]

In [16]:
unique_columns_list = filtered_df_based_on_similar_names.columns.tolist()

In [17]:
unique_columns_list

['cds',
 'rtype',
 'schoolname',
 'districtname',
 'countyname',
 'charter_flag',
 'coe_flag',
 'dass_flag',
 'studentgroup',
 'currdenom',
 'currstatus',
 'priordenom',
 'priorstatus',
 'change',
 'statuslevel',
 'changelevel',
 'color',
 'box',
 'currnsizemet',
 'priornsizemet',
 'accountabilitymet',
 'hscutpoints',
 'pairshare_method',
 'currprate_enrolled',
 'currprate_tested',
 'currprate',
 'currnumPRLOSS',
 'currdenom_withoutPRLOSS',
 'currstatus_withoutPRLOSS',
 'priorprate_enrolled',
 'priorprate_tested',
 'priorprate',
 'priornumPRLOSS',
 'priordenom_withoutPRLOSS',
 'priorstatus_withoutPRLOSS',
 'indicator',
 'reportingyear']

In [18]:
filtered_df_based_on_similar_names = filtered_df_based_on_similar_names[['schoolname', 'studentgroup', 'statuslevel']]
filtered_df_based_on_similar_names

,schoolname,studentgroup,statuslevel
132298,Jordan (June) School for Equity,AA,0
132299,Jordan (June) School for Equity,ALL,1
132300,Jordan (June) School for Equity,AS,0
132301,Jordan (June) School for Equity,EL,1
132302,Jordan (June) School for Equity,ELO,1
...,...,...,...
132770,Washington (George) High,RFP,2
132771,Washington (George) High,SBA,4
132772,Washington (George) High,SED,3
132773,Washington (George) High,SWD,1


In [19]:
overallperformance = filtered_df_based_on_similar_names[filtered_df_based_on_similar_names['studentgroup'].str.contains('ALL')]

In [20]:
overallperformance

,schoolname,studentgroup,statuslevel
132299,Jordan (June) School for Equity,ALL,1
132316,City Arts & Leadership Academy,ALL,2
132346,Five Keys Independence HS (SF Sheriff's),ALL,1
132361,S.F. International High,ALL,1
132495,Wells (Ida B.) High,ALL,1
132511,Downtown High,ALL,1
132529,Independence High,ALL,1
132547,Wallenberg (Raoul) Traditional High,ALL,2
132565,Burton (Phillip and Sala) Academic High,ALL,3
132583,Balboa High,ALL,2


In [21]:
overallperformance.sort_values('statuslevel')

,schoolname,studentgroup,statuslevel
132299,Jordan (June) School for Equity,ALL,1
132346,Five Keys Independence HS (SF Sheriff's),ALL,1
132361,S.F. International High,ALL,1
132495,Wells (Ida B.) High,ALL,1
132529,Independence High,ALL,1
132511,Downtown High,ALL,1
132619,Marshall (Thurgood) High,ALL,1
132635,Life Learning Academy Charter,ALL,1
132722,Mission High,ALL,1
132741,O'Connell (John) High,ALL,1


In [22]:
df2 = pd.read_csv('mathdownload2024 (1).csv')

/tmp/ipykernel_172/3118812136.py:1: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  df2 = pd.read_csv('mathdownload2024 (1).csv')


In [23]:
df_clean2 = df2.dropna(subset=['countyname'])
schools2 = df_clean2[df_clean2['countyname'].str.contains('San Francisco')]

In [24]:
schools2

,cds,rtype,schoolname,districtname,countyname,charter_flag,coe_flag,dass_flag,studentgroup,currdenom,...,currdenom_withoutPRLOSS,currstatus_withoutPRLOSS,priorprate_enrolled,priorprate_tested,priorprate,priornumPRLOSS,priordenom_withoutPRLOSS,priorstatus_withoutPRLOSS,indicator,reportingyear
132348,38103890000000,D,NaN,San Francisco County Office of Education,San Francisco,NaN,Y,NaN,AA,22,...,2,NaN,15.0,3.0,20.0,12.0,1,NaN,MATH,2024
132349,38103890000000,D,NaN,San Francisco County Office of Education,San Francisco,NaN,Y,NaN,ALL,166,...,13,-183.5,71.0,19.0,27.0,49.0,12,-169.8,MATH,2024
132350,38103890000000,D,NaN,San Francisco County Office of Education,San Francisco,NaN,Y,NaN,AS,16,...,1,NaN,5.0,2.0,40.0,3.0,1,NaN,MATH,2024
132351,38103890000000,D,NaN,San Francisco County Office of Education,San Francisco,NaN,Y,NaN,CAA,1,...,1,NaN,NaN,NaN,NaN,NaN,0,NaN,MATH,2024
132352,38103890000000,D,NaN,San Francisco County Office of Education,San Francisco,NaN,Y,NaN,EL,40,...,2,NaN,19.0,8.0,42.0,10.0,5,NaN,MATH,2024
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
134378,38771310137307,S,KIPP Bayview Elementary,SBE - KIPP Bayview Elementary,San Francisco,Y,NaN,NaN,RFP,2,...,2,NaN,2.0,2.0,100.0,NaN,2,NaN,MATH,2024
134379,38771310137307,S,KIPP Bayview Elementary,SBE - KIPP Bayview Elementary,San Francisco,Y,NaN,NaN,SBA,45,...,45,-77.9,NaN,NaN,NaN,NaN,56,-99.6,MATH,2024
134380,38771310137307,S,KIPP Bayview Elementary,SBE - KIPP Bayview Elementary,San Francisco,Y,NaN,NaN,SED,43,...,43,-75.2,51.0,51.0,100.0,NaN,50,-100.1,MATH,2024
134381,38771310137307,S,KIPP Bayview Elementary,SBE - KIPP Bayview Elementary,San Francisco,Y,NaN,NaN,SWD,11,...,11,-121.0,11.0,11.0,100.0,NaN,11,-132.9,MATH,2024


In [25]:
highschools2 = schools2[~schools2['schoolname'].str.contains('Elementary|Middle', case=False, na=False)]

In [26]:
onlyhighschools2 = pd.read_csv('public_high_schools.csv')

In [29]:
names2 = onlyhighschools2['Campus Name'].tolist()

In [30]:
similar_names2 = highschools2['schoolname'].apply(lambda x: find_similar_names(x, names2))

Checking 'S.F. County Court Woodside Learning Ctr' against 'Life Learning Academy Charter School' with score 40.0
Checking 'S.F. County Court Woodside Learning Ctr' against 'Life Learning Academy Charter School' with score 40.0
Checking 'S.F. County Court Woodside Learning Ctr' against 'Life Learning Academy Charter School' with score 40.0
Checking 'S.F. County Court Woodside Learning Ctr' against 'Life Learning Academy Charter School' with score 40.0
Checking 'S.F. County Court Woodside Learning Ctr' against 'Life Learning Academy Charter School' with score 40.0
Checking 'S.F. County Court Woodside Learning Ctr' against 'Life Learning Academy Charter School' with score 40.0
Checking 'S.F. County Court Woodside Learning Ctr' against 'Life Learning Academy Charter School' with score 40.0
Checking 'S.F. County Opportunity (Hilltop)' against 'City Arts And Tech High School' with score 34.92063492063492
Checking 'S.F. County Opportunity (Hilltop)' against 'City Arts And Tech High School' w

In [32]:
filtered_df_based_on_similar_names2 =highschools2[similar_names2]

In [33]:
filtered_df_based_on_similar_names2['schoolname'].unique()

array(['Jordan (June) School for Equity',
       'City Arts & Leadership Academy',
       "Five Keys Independence HS (SF Sheriff's)",
       'S.F. International High', 'San Francisco Public Montessori',
       'Wells (Ida B.) High', 'Downtown High', 'Independence High',
       'Wallenberg (Raoul) Traditional High',
       'Burton (Phillip and Sala) Academic High', 'Balboa High',
       'Marshall (Thurgood) High', 'Life Learning Academy Charter',
       'Gateway High', 'Galileo High', 'Lincoln (Abraham) High',
       'Mission High', "O'Connell (John) High",
       'Washington (George) High', 'San Francisco Community Alternative'],
      dtype=object)

In [35]:
schools_to_remove2 = [
    'San Francisco Public Montessori',
    'San Francisco Community Alternative'
]

In [36]:
filtered_df_based_on_similar_names2 = filtered_df_based_on_similar_names2[
    ~filtered_df_based_on_similar_names2['schoolname'].isin(schools_to_remove2)
]

In [37]:
unique_columns_list2 = filtered_df_based_on_similar_names2.columns.tolist()

In [38]:
filtered_df_based_on_similar_names2 = filtered_df_based_on_similar_names2[['schoolname', 'studentgroup', 'statuslevel']]


In [39]:
overallperformance2 = filtered_df_based_on_similar_names2[filtered_df_based_on_similar_names2['studentgroup'].str.contains('ALL')]

In [40]:
overallperformance2.sort_values('statuslevel')

,schoolname,studentgroup,statuslevel
132480,Jordan (June) School for Equity,ALL,1
132497,City Arts & Leadership Academy,ALL,1
132527,Five Keys Independence HS (SF Sheriff's),ALL,1
132542,S.F. International High,ALL,1
132676,Wells (Ida B.) High,ALL,1
132692,Downtown High,ALL,1
132710,Independence High,ALL,1
132815,Life Learning Academy Charter,ALL,1
132902,Mission High,ALL,1
132799,Marshall (Thurgood) High,ALL,1


In [44]:
overallperformance.info()

<class 'pandas.core.frame.DataFrame'>
Index: 18 entries, 132299 to 132758
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   schoolname    18 non-null     object
 1   studentgroup  18 non-null     object
 2   statuslevel   18 non-null     int64 
dtypes: int64(1), object(2)
memory usage: 576.0+ bytes


In [45]:
overallperformance2.info()

<class 'pandas.core.frame.DataFrame'>
Index: 18 entries, 132480 to 132938
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   schoolname    18 non-null     object
 1   studentgroup  18 non-null     object
 2   statuslevel   18 non-null     int64 
dtypes: int64(1), object(2)
memory usage: 576.0+ bytes


In [43]:
combined_score

132299   NaN
132316   NaN
132346   NaN
132361   NaN
132480   NaN
132495   NaN
132497   NaN
132511   NaN
132527   NaN
132529   NaN
132542   NaN
132547   NaN
132565   NaN
132583   NaN
132619   NaN
132635   NaN
132648   NaN
132664   NaN
132676   NaN
132683   NaN
132692   NaN
132710   NaN
132722   NaN
132728   NaN
132741   NaN
132746   NaN
132758   NaN
132763   NaN
132799   NaN
132815   NaN
132828   NaN
132844   NaN
132863   NaN
132902   NaN
132921   NaN
132938   NaN
Name: statuslevel, dtype: float64